# 001 — Исследование результатов эксперимента

Открывает таблицу `parameter_performance_by_slice` (её пишет `run-experiment`, AN-1)
и строит базовую аналитику: рейтинг параметров (AN-2), вектор стабильности (AN-3)
и heatmap (AN-4).

Путь к результатам берётся из переменной окружения `TR_RESULTS`. Если файла нет,
ноутбук использует синтетические данные — поэтому исполняется сверху донизу без правок.

In [ ]:
import os
import tempfile
from datetime import UTC, datetime
from pathlib import Path

import polars as pl

from trading_research.analytics.plots import build_heatmap, plot_heatmap
from trading_research.analytics.queries import (
    PerformanceRecord,
    build_parameter_performance,
    load_performance_table,
    rank_params,
)
from trading_research.analytics.stability import stability_vector
from trading_research.backtest.metrics import Metrics
from trading_research.domain import TimeSlice, TimeSliceKind

In [ ]:
def _synthetic_performance() -> pl.DataFrame:
    """Мини-таблица результатов, чтобы ноутбук работал без прогона."""

    def mk(total: float) -> Metrics:
        return Metrics(total, -5.0, 1.0, 1.0, 2.0, 55.0, 0.3, 1.1, 2.0, 7, 8760.0, 30, True)

    def sl(sid: str) -> TimeSlice:
        m = int(sid.split("-")[1])
        return TimeSlice(
            id=sid,
            start=datetime(2024, m, 1, tzinfo=UTC),
            end=datetime(2024, m + 1, 1, tzinfo=UTC),
            label=sid,
            kind=TimeSliceKind.MONTH,
        )

    recs = []
    grid = {(5, 20): [10.0, 20.0], (10, 50): [5.0, 15.0], (5, 50): [-5.0, 2.0]}
    for (fast, slow), totals in grid.items():
        for sid, total in zip(["2024-01", "2024-02"], totals, strict=True):
            recs.append(
                PerformanceRecord(
                    "demo",
                    "ma_cross",
                    "BTCUSDT",
                    "1h",
                    sl(sid),
                    {"fast_ma": fast, "slow_ma": slow},
                    mk(total),
                )
            )
    return build_parameter_performance(recs)


results_path = os.environ.get("TR_RESULTS")
if results_path and Path(results_path).exists():
    perf = load_performance_table(results_path)
else:
    perf = _synthetic_performance()

perf.head()

## Рейтинг параметров (AN-2)

In [ ]:
ranking = rank_params(perf, metric="total_return_pct", top=10)
ranking

## Вектор стабильности (AN-3)

In [ ]:
stability = stability_vector(perf, top_quantile=0.5)
stability.sort("stability_score", descending=True)

## Heatmap параметров (AN-4)

In [ ]:
heatmap = build_heatmap(perf, x="fast_ma", y="slow_ma", metric="total_return_pct")
out = Path(tempfile.mkdtemp()) / "heatmap.png"
plot_heatmap(heatmap, out)
print("heatmap saved to", out)